# 03 · Estabilidad multisemilla y auditoría estructural V8

Este cuaderno **no reentrena** y nunca abre el bloque externo. Audita que la
adaptación estructural no sea nula, calcula estabilidad Jaccard y Nogueira
usando el universo candidato completo, y agrega separadamente:

- grafos predictivos Exact-K;
- redes biológicas diferenciales Exact-K;
- backbone biológico compartido.

Se distinguen dos núcleos génicos que no deben confundirse: presencia del
nodo en 5/5 redes y genes incidentes en una misma arista retenida 5/5. El
segundo es el panel primario para afirmaciones mecanísticas. La estabilidad
entre semillas mide reproducibilidad algorítmica con datos fijos; no
representa estabilidad frente a nuevos pacientes.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro de InterGATE_V8.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v8_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V8"
RESULTS = PROJECT_ROOT / "results_v8"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
from itertools import combinations
from scipy.stats import spearmanr

from intergate.biology import aggregate_biology_seed_tables
from intergate.reviewer_analysis import (
    consensus_from_seed_tables, topology_metrics,
)
from intergate.stability import pairwise_jaccard_stats, nogueira_stability

runtime = prepare_runtime(CFG, include_external=False)
assert runtime["external_opened"] is False
SEEDS = tuple(map(int, CFG.STABILITY_SEEDS))
if len(SEEDS) != 5 or len(set(SEEDS)) != 5:
    raise ValueError("El protocolo requiere exactamente cinco semillas distintas.")

AGG = RESULTS / "03_stability"
BIO_AGG = AGG / "biology_consensus"
PRED_AGG = AGG / "predictive_consensus"
SHARED_AGG = AGG / "shared_backbone_consensus"
AGG.mkdir(parents=True, exist_ok=True)
BIO_AGG.mkdir(parents=True, exist_ok=True)
PRED_AGG.mkdir(parents=True, exist_ok=True)
SHARED_AGG.mkdir(parents=True, exist_ok=True)

def safe_name(value):
    return "".join(
        char if char.isalnum() or char in "-_" else "_"
        for char in str(value)
    )

edge_index = np.asarray(runtime["edge_index"], dtype=np.int64)
genes = np.asarray(list(map(str, runtime["genes"])), dtype=object)
candidate_edge_universe = {
    (str(genes[source]), str(genes[target]))
    for source, target in edge_index.T
    if int(source) != int(target)
}
candidate_gene_universe = {
    gene for edge in candidate_edge_universe for gene in edge
}
print({
    "candidate_directed_edges_without_self_loops": len(candidate_edge_universe),
    "candidate_connected_genes": len(candidate_gene_universe),
    "external_opened": runtime["external_opened"],
})


In [ ]:
# 1) La adaptación leave-one-development-cohort-out debe cambiar parámetros
# y rankings. Un Jaccard alto no se interpreta como robustez si los deltas
# de parámetros/logits son exactamente cero.
audits = []
topology_rows = []
for seed in SEEDS:
    seed_dir = RUN_ROOT / f"seed_{seed}"
    audit = pd.read_csv(seed_dir / "transport_adaptation_audit.csv")
    audit.insert(0, "seed", seed)
    audits.append(audit)
    summary = json.loads(
        (seed_dir / "transport_support_summary.json").read_text(encoding="utf-8")
    )
    if summary["adaptation_audit"]["all_parameter_deltas_zero"]:
        raise RuntimeError(f"Semilla {seed}: adaptación estructural nula.")
    for class_name, details in summary["classes"].items():
        for graph_kind in ("raw_differential", "module_aware"):
            topology_rows.append({
                "seed": seed, "class_name": class_name,
                "graph_kind": graph_kind,
                **details["biology_topology"][graph_kind],
            })
audit_all = pd.concat(audits, ignore_index=True)
audit_all.to_csv(AGG / "adaptation_audit_all_seeds.csv.gz", index=False, compression="gzip")
logit_audit = audit_all[audit_all.parameter.eq("__LOGITS__")]
final_delta = audit_all[
    audit_all.parameter.astype(str).str.startswith("__FINAL__:")
]
fold_total_delta = final_delta.groupby(
    ["seed", "heldout_cohort"]
).step_delta_l2.sum()
audit_summary = pd.DataFrame([{
    "n_seeds": len(SEEDS),
    "min_fold_total_parameter_delta_l2": float(fold_total_delta.min()),
    "median_mean_abs_logit_delta": float(logit_audit.mean_abs_logit_delta.median()),
    "median_rank_spearman": float(logit_audit.rank_spearman.median()),
    "median_topk_jaccard_base_vs_adapted": float(logit_audit.base_adapted_topk_jaccard.median()),
    "total_topk_turnover_edges": int(logit_audit.topk_turnover_edges.sum()),
}])
display(audit_summary)
if audit_summary.min_fold_total_parameter_delta_l2.iloc[0] <= 0:
    raise RuntimeError("Existe una deleción de cohorte sin adaptación.")
audit_summary.to_csv(AGG / "adaptation_audit_summary.csv", index=False)


In [ ]:
# 2) Consenso predictivo con estabilidad corregida por azar. El universo
# debe ser el prior candidato completo, no la unión de las cinco selecciones.
predictive = consensus_from_seed_tables(
    RUN_ROOT,
    frequency_threshold=0.80,
    strict_threshold=1.00,
    expected_runs=5,
    expected_seed_values=SEEDS,
    edge_universe=candidate_edge_universe,
    gene_universe=candidate_gene_universe,
    output_dir=PRED_AGG,
)

predictive_rows = []
for class_name, payload in predictive.items():
    predictive_rows.append({
        "class_name": class_name,
        "k_edges": int(payload["k"]),
        "mean_pairwise_edge_jaccard": float(
            payload["edge_jaccard_stats"]["jaccard_mean"]
        ),
        "sd_pairwise_edge_jaccard": float(
            payload["edge_jaccard_stats"]["jaccard_std"]
        ),
        "mean_pairwise_gene_jaccard": float(
            payload["gene_jaccard_stats"]["jaccard_mean"]
        ),
        "edge_nogueira": float(payload["edge_nogueira_stability"]),
        "gene_nogueira": float(payload["gene_nogueira_stability"]),
        "edge_core_4of5": int(len(payload["stable_edges_80"])),
        "edge_core_5of5": int(len(payload["stable_edges_100"])),
        "node_presence_core_5of5": int(len(payload["gene_core100"])),
        "edge_supported_gene_core_5of5": int(
            len(payload["edge_core100_genes"])
        ),
    })
predictive_summary = pd.DataFrame(predictive_rows)
predictive_summary.to_csv(
    PRED_AGG / "predictive_stability_summary.csv", index=False
)
display(predictive_summary)


In [ ]:
# 3) Agregación biológica Exact-K. La función base construye el consenso
# modular; a continuación se añaden dirección, estabilidad Nogueira y las
# dos definiciones explícitas de núcleo génico.
biological = aggregate_biology_seed_tables(
    RUN_ROOT,
    seeds=SEEDS,
    class_names=runtime["classes"],
    output_dir=BIO_AGG,
    pool_multiplier=CFG.BIOLOGY_MODULE_POOL_MULTIPLIER,
    target_components=CFG.BIOLOGY_MODULE_TARGET_COMPONENTS,
    min_component_edges=CFG.BIOLOGY_MODULE_MIN_EDGES,
)

biology_consensus_sets = {}
biology_node_presence_cores = {}
biology_edge_supported_cores = {}
frequency_rows = []
biology_rows = []

for class_name in runtime["classes"]:
    safe = safe_name(class_name)
    frames = []
    for seed in SEEDS:
        path = RUN_ROOT / f"seed_{seed}" / f"biology_edges_{safe}.csv"
        frame = pd.read_csv(path)
        frame["seed"] = int(seed)
        frames.append(frame)

    edge_sets = [
        set(zip(frame.source_gene.astype(str), frame.target_gene.astype(str)))
        for frame in frames
    ]
    gene_sets = [
        {gene for edge in selected for gene in edge}
        for selected in edge_sets
    ]
    union = pd.concat(frames, ignore_index=True)
    edge_stability = (
        union.groupby(["source_gene", "target_gene"], as_index=False)
        .agg(
            selection_count=("seed", "nunique"),
            median_differential_logit=("differential_logit", "median"),
            median_biology_score=("biology_score", "median"),
            mean_context_score=("context_score", "mean"),
            mean_transport_support=("selection_frequency", "mean"),
            candidate_edge_id=("candidate_edge_id", "min"),
            prior_weight=("prior_weight", "median"),
            edge_type=("edge_type", "first"),
            strengthened_count=(
                "differential_logit", lambda values: int((values > 0).sum())
            ),
            attenuated_count=(
                "differential_logit", lambda values: int((values < 0).sum())
            ),
        )
    )
    edge_stability["selection_frequency_seed"] = (
        edge_stability.selection_count / float(len(SEEDS))
    )
    edge_stability["consensus_direction"] = np.select(
        [
            edge_stability.median_differential_logit.gt(0),
            edge_stability.median_differential_logit.lt(0),
        ],
        ["strengthened_for_class", "attenuated_for_class"],
        default="undetermined",
    )
    edge_stability["direction_agreement"] = (
        edge_stability[["strengthened_count", "attenuated_count"]].max(axis=1)
        / edge_stability.selection_count.clip(lower=1)
    )
    edge_stability["strict_edge_5of5"] = edge_stability.selection_count.eq(
        len(SEEDS)
    )
    edge_stability = edge_stability.sort_values(
        ["selection_count", "median_biology_score", "candidate_edge_id"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    edge_stability.to_csv(
        BIO_AGG / f"biology_edge_stability_{safe}.csv", index=False
    )

    consensus_path = BIO_AGG / f"biology_consensus_{safe}.csv"
    consensus = pd.read_csv(consensus_path)
    consensus = consensus.drop(
        columns=[
            "median_differential_logit", "consensus_direction",
            "direction_agreement", "strengthened_count",
            "attenuated_count", "strict_edge_5of5",
        ],
        errors="ignore",
    ).merge(
        edge_stability[[
            "source_gene", "target_gene", "median_differential_logit",
            "consensus_direction", "direction_agreement",
            "strengthened_count", "attenuated_count", "strict_edge_5of5",
        ]],
        on=["source_gene", "target_gene"], how="left", validate="one_to_one",
    )
    consensus.to_csv(consensus_path, index=False)
    consensus_set = set(zip(
        consensus.source_gene.astype(str), consensus.target_gene.astype(str)
    ))
    biology_consensus_sets[class_name] = consensus_set

    strict_edges = set(zip(
        edge_stability.loc[edge_stability.strict_edge_5of5, "source_gene"].astype(str),
        edge_stability.loc[edge_stability.strict_edge_5of5, "target_gene"].astype(str),
    ))
    edge_supported = sorted({gene for edge in strict_edges for gene in edge})
    gene_table = pd.read_csv(
        BIO_AGG / f"biology_gene_stability_{safe}.csv"
    )
    node_presence = sorted(
        gene_table.loc[
            pd.to_numeric(gene_table.selection_count).eq(len(SEEDS)), "gene"
        ].astype(str).str.upper().unique()
    )
    biology_node_presence_cores[class_name] = node_presence
    biology_edge_supported_cores[class_name] = edge_supported

    for count, n_edges in edge_stability.selection_count.value_counts().items():
        frequency_rows.append({
            "class_name": class_name,
            "selection_count": int(count),
            "n_union_edges": int(n_edges),
        })

    edge_jaccard = pairwise_jaccard_stats(edge_sets)
    gene_jaccard = pairwise_jaccard_stats(gene_sets)
    strict_direction = edge_stability.loc[
        edge_stability.strict_edge_5of5, "direction_agreement"
    ]
    row = {
        "class_name": class_name,
        "k_edges": int(len(frames[0])),
        "n_union_edges": int(len(edge_stability)),
        "mean_pairwise_edge_jaccard": float(edge_jaccard["jaccard_mean"]),
        "sd_pairwise_edge_jaccard": float(edge_jaccard["jaccard_std"]),
        "mean_pairwise_gene_jaccard": float(gene_jaccard["jaccard_mean"]),
        "edge_nogueira": float(
            nogueira_stability(edge_sets, candidate_edge_universe)
        ),
        "gene_nogueira": float(
            nogueira_stability(gene_sets, candidate_gene_universe)
        ),
        "strict_edges_5of5": int(len(strict_edges)),
        "node_presence_core_5of5": int(len(node_presence)),
        "edge_supported_gene_core_5of5": int(len(edge_supported)),
        "mean_direction_agreement_strict_edges": float(
            strict_direction.mean() if len(strict_direction) else np.nan
        ),
    }
    biology_rows.append(row)
    biological[class_name].update(row)

biology_summary = pd.DataFrame(biology_rows)
biology_summary.to_csv(
    BIO_AGG / "biology_stability_summary.csv", index=False
)
pd.DataFrame(frequency_rows).sort_values(
    ["class_name", "selection_count"]
).to_csv(
    BIO_AGG / "biology_selection_frequency_distribution.csv", index=False
)
(BIO_AGG / "biology_stability_summary.json").write_text(
    json.dumps(biological, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(biology_summary)


In [ ]:
# 4) Backbone compartido: consenso multisemilla independiente de las redes
# diferenciales de clase.
shared_frames = []
for seed in SEEDS:
    frame = pd.read_csv(
        RUN_ROOT / f"seed_{seed}" / "biology_shared_backbone.csv"
    )
    frame["seed"] = int(seed)
    frame["within_seed_rank"] = np.arange(1, len(frame) + 1)
    shared_frames.append(frame)
shared_sizes = [len(frame) for frame in shared_frames]
if len(set(shared_sizes)) != 1:
    raise RuntimeError(f"Backbone compartido no es Exact-K: {shared_sizes}")
shared_k = int(shared_sizes[0])
shared_sets = [
    set(zip(frame.source_gene.astype(str), frame.target_gene.astype(str)))
    for frame in shared_frames
]
shared_gene_sets = [
    {gene for edge in selected for gene in edge} for selected in shared_sets
]
shared_union = pd.concat(shared_frames, ignore_index=True)
shared_stability = (
    shared_union.groupby(["source_gene", "target_gene"], as_index=False)
    .agg(
        selection_count=("seed", "nunique"),
        median_shared_score=("shared_score", "median"),
        median_within_seed_rank=("within_seed_rank", "median"),
        candidate_edge_id=("candidate_edge_id", "min"),
        prior_weight=("prior_weight", "median"),
        edge_type=("edge_type", "first"),
    )
)
shared_stability["selection_frequency_seed"] = (
    shared_stability.selection_count / float(len(SEEDS))
)
shared_stability["strict_edge_5of5"] = shared_stability.selection_count.eq(
    len(SEEDS)
)
shared_stability = shared_stability.sort_values(
    ["selection_count", "median_within_seed_rank", "median_shared_score", "candidate_edge_id"],
    ascending=[False, True, False, True],
).reset_index(drop=True)
shared_stability["in_exact_k_consensus"] = False
shared_stability.loc[: shared_k - 1, "in_exact_k_consensus"] = True
shared_consensus = shared_stability.iloc[:shared_k].copy()
shared_consensus.insert(0, "consensus_rank", np.arange(1, shared_k + 1))
shared_stability.to_csv(
    SHARED_AGG / "shared_backbone_edge_stability.csv", index=False
)
shared_consensus.to_csv(
    SHARED_AGG / "shared_backbone_consensus.csv", index=False
)

shared_gene_counts = {}
for selected in shared_gene_sets:
    for gene in selected:
        shared_gene_counts[gene] = shared_gene_counts.get(gene, 0) + 1
shared_gene_stability = pd.DataFrame({
    "gene": sorted(shared_gene_counts),
    "selection_count": [
        shared_gene_counts[gene] for gene in sorted(shared_gene_counts)
    ],
})
shared_gene_stability["selection_frequency"] = (
    shared_gene_stability.selection_count / float(len(SEEDS))
)
shared_gene_stability["node_presence_core_5of5"] = (
    shared_gene_stability.selection_count.eq(len(SEEDS))
)
shared_strict_edges = set(zip(
    shared_stability.loc[shared_stability.strict_edge_5of5, "source_gene"].astype(str),
    shared_stability.loc[shared_stability.strict_edge_5of5, "target_gene"].astype(str),
))
shared_edge_supported_genes = sorted({
    gene for edge in shared_strict_edges for gene in edge
})
shared_gene_stability["edge_supported_core_5of5"] = (
    shared_gene_stability.gene.isin(shared_edge_supported_genes)
)
shared_gene_stability.to_csv(
    SHARED_AGG / "shared_backbone_gene_stability.csv", index=False
)
shared_pairwise_rows = []
for left, right in combinations(range(len(SEEDS)), 2):
    union = shared_sets[left] | shared_sets[right]
    shared_pairwise_rows.append({
        "seed_a": int(SEEDS[left]), "seed_b": int(SEEDS[right]),
        "edge_jaccard": float(
            len(shared_sets[left] & shared_sets[right]) / len(union)
            if union else np.nan
        ),
    })
pd.DataFrame(shared_pairwise_rows).to_csv(
    SHARED_AGG / "shared_backbone_pairwise_jaccard.csv", index=False
)
shared_summary = {
    "k_edges": shared_k,
    "n_union_edges": int(len(shared_stability)),
    "mean_pairwise_edge_jaccard": float(
        pairwise_jaccard_stats(shared_sets)["jaccard_mean"]
    ),
    "edge_nogueira": float(
        nogueira_stability(shared_sets, candidate_edge_universe)
    ),
    "gene_nogueira": float(
        nogueira_stability(shared_gene_sets, candidate_gene_universe)
    ),
    "strict_edges_5of5": int(len(shared_strict_edges)),
    "node_presence_core_5of5": int(
        shared_gene_stability.node_presence_core_5of5.sum()
    ),
    "edge_supported_gene_core_5of5": int(len(shared_edge_supported_genes)),
}
(SHARED_AGG / "shared_backbone_summary.json").write_text(
    json.dumps(shared_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame([shared_summary]))


In [ ]:
# 5) Solapamiento entre clases y alineación del logit diferencial con la
# evidencia contextual, calculada sobre todas las aristas candidatas.
overlap_rows = []
class_names = list(runtime["classes"])
for left, right in combinations(class_names, 2):
    for graph_family, graph_sets in (
        ("predictive_exact_k", {
            name: predictive[name]["exact_edges"] for name in class_names
        }),
        ("biology_exact_k", biology_consensus_sets),
    ):
        a, b = graph_sets[left], graph_sets[right]
        union = a | b
        overlap_rows.append({
            "graph_family": graph_family,
            "class_a": left, "class_b": right,
            "intersection_edges": int(len(a & b)),
            "union_edges": int(len(union)),
            "jaccard": float(len(a & b) / len(union) if union else np.nan),
        })
class_overlap = pd.DataFrame(overlap_rows)
class_overlap.to_csv(AGG / "class_graph_overlap.csv", index=False)

context_rows = []
for seed in SEEDS:
    dense = np.load(
        RUN_ROOT / f"seed_{seed}" / "transport_support_dense.npz",
        allow_pickle=True,
    )
    delta = np.asarray(dense["differential_logits"], dtype=float)
    context = np.asarray(dense["context_score"], dtype=float)
    support = np.asarray(dense["selection_frequency"], dtype=float)
    for class_id, class_name in enumerate(class_names):
        usable = np.isfinite(delta[class_id]) & np.isfinite(context[class_id])
        rho, p_value = spearmanr(
            delta[class_id, usable], context[class_id, usable]
        )
        rho_support, p_support = spearmanr(
            np.abs(delta[class_id, usable]), support[class_id, usable]
        )
        context_rows.append({
            "seed": int(seed), "class_name": class_name,
            "n_candidate_edges": int(usable.sum()),
            "spearman_delta_vs_context": float(rho),
            "p_delta_vs_context": float(p_value),
            "spearman_abs_delta_vs_transport_support": float(rho_support),
            "p_abs_delta_vs_transport_support": float(p_support),
        })
context_alignment = pd.DataFrame(context_rows)
context_alignment.to_csv(
    AGG / "context_alignment_all_candidate_edges.csv", index=False
)
display(class_overlap)
display(
    context_alignment.groupby("class_name")[[
        "spearman_delta_vs_context",
        "spearman_abs_delta_vs_transport_support",
    ]].agg(["mean", "std"])
)


In [ ]:
# 6) Topología y paneles para el notebook 05. `strict_5of5` se mantiene
# como alias compatible, pero ahora apunta al núcleo soportado por las
# mismas aristas 5/5, no a la mera presencia del nodo.
topology = pd.DataFrame(topology_rows)
topology.to_csv(AGG / "biology_topology_raw_vs_module.csv", index=False)
topology_summary = (
    topology.groupby(["class_name", "graph_kind"])[
        ["n_edges", "n_nodes", "n_components", "giant_fraction", "dyad_components", "degree_one_nodes"]
    ].mean().reset_index()
)
display(topology_summary)

consensus_topology_rows = []
for class_name, selected in biology_consensus_sets.items():
    consensus_topology_rows.append({
        "class_name": class_name, "graph_set": "frequency_ranked_exact_k",
        **topology_metrics(selected),
    })
    strict_table = pd.read_csv(
        BIO_AGG / f"biology_edge_stability_{safe_name(class_name)}.csv"
    )
    strict_set = set(zip(
        strict_table.loc[strict_table.strict_edge_5of5, "source_gene"].astype(str),
        strict_table.loc[strict_table.strict_edge_5of5, "target_gene"].astype(str),
    ))
    consensus_topology_rows.append({
        "class_name": class_name, "graph_set": "strict_edge_core_5of5",
        **topology_metrics(strict_set),
    })
consensus_topology = pd.DataFrame(consensus_topology_rows)
consensus_topology.to_csv(
    AGG / "biology_consensus_topology.csv", index=False
)
display(consensus_topology)

panels = {
    "protocol": {
        "primary_definition": (
            "genes incident to at least one identical directed edge selected "
            "in all five module-aware Exact-K networks"
        ),
        "secondary_definition": (
            "nodes present in all five networks, possibly through different edges"
        ),
        "seeds": list(SEEDS),
        "external_opened": False,
    },
    "strict_5of5": biology_edge_supported_cores,
    "edge_supported_gene_core_5of5": biology_edge_supported_cores,
    "node_presence_core_5of5": biology_node_presence_cores,
    "shared_backbone": {
        "edge_supported_gene_core_5of5": shared_edge_supported_genes,
        "node_presence_core_5of5": sorted(
            shared_gene_stability.loc[
                shared_gene_stability.node_presence_core_5of5, "gene"
            ].astype(str)
        ),
    },
}
(AGG / "biology_panels_v8.json").write_text(
    json.dumps(panels, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Auditoría y agregación V8 completas:", AGG)
print("El bloque externo permanece cerrado.")
